# Colab GPU Experiment Runner

This notebook is intentionally thin. Most Colab logic now lives in `scripts/06_colab_session.py`.

Typical usage:
- leave `MODE = "auto"` and run all cells in one Colab session
- open more sessions with `MODE = "worker"` for parallel job claiming
- use the final status cell to inspect shared run state

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path('/content/drive/MyDrive/carbon-aware-recsys')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = '6-neumfs-sharp-curve'

MODE = 'auto'  # choose: 'auto', 'prepare', 'worker', 'finalize'
WORKER_NAME = None  # optional: e.g. 'colab-gpu-1'

SYNC_REPO = True
INSTALL_DEPS = True
VERIFY_RUNTIME = True
RETRY_FAILED_JOBS = True

CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF', 'LightGCN']

TOP_K_CANDIDATES = 100
TOP_K_RERANK = 10
LEARNING_RATE = 1e-3
EPOCHS = 50
EVAL_STEP = 10
MAX_USERS = None

PREPARE_CARBON = False
FORCE = False
FORCE_CARBON = False
SKIP_LLM = False
LLM_CACHE_ONLY = False
FINALIZE_WHEN_DONE = MODE == 'worker'

if SYNC_REPO and not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)
    ], check=True)

if SYNC_REPO:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

git_commit = subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print({
    'project_root': str(PROJECT_ROOT),
    'repo_branch': REPO_BRANCH,
    'git_commit': git_commit,
    'mode': MODE,
})

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    'scripts/06_colab_session.py',
    '--project-root', str(PROJECT_ROOT),
    '--repo-branch', REPO_BRANCH,
    '--mode', MODE,
    '--categories', *CATEGORIES,
    '--models', *MODELS,
    '--top-k-candidates', str(TOP_K_CANDIDATES),
    '--top-k-rerank', str(TOP_K_RERANK),
    '--learning-rate', str(LEARNING_RATE),
    '--epochs', str(EPOCHS),
    '--eval-step', str(EVAL_STEP),
]

if SYNC_REPO:
    cmd.append('--sync-repo')
if INSTALL_DEPS:
    cmd.append('--install')
if VERIFY_RUNTIME:
    cmd.append('--verify-runtime')
if RETRY_FAILED_JOBS:
    cmd.append('--retry-failed-jobs')
if WORKER_NAME:
    cmd.extend(['--worker-name', WORKER_NAME])
if MAX_USERS is not None:
    cmd.extend(['--max-users', str(MAX_USERS)])
if PREPARE_CARBON:
    cmd.append('--prepare-carbon')
if FORCE:
    cmd.append('--force')
if FORCE_CARBON:
    cmd.append('--force-carbon')
if SKIP_LLM:
    cmd.append('--skip-llm')
if LLM_CACHE_ONLY:
    cmd.append('--llm-cache-only')
if MODE == 'worker' and FINALIZE_WHEN_DONE:
    cmd.append('--finalize-when-done')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

In [ ]:
import subprocess
import sys

status_cmd = [
    sys.executable,
    'scripts/06_colab_session.py',
    '--project-root', str(PROJECT_ROOT),
    '--mode', 'status',
]
subprocess.run(status_cmd, cwd=str(PROJECT_ROOT), check=True)